# CROCO Processing — input-noise robustness on Algerian (copy of notebook_nuredduna_seeds)

## 0. Setup

In [ ]:
import sys
import os
import warnings
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm
from scipy.interpolate import interp1d
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import joblib

warnings.filterwarnings('ignore')

# Path to the local xcroco package (CROCO python tools), needed to import
# the model/grid-operations helpers used throughout this notebook.
XCROCO_PATH = os.path.expanduser('./croco_pytools-v2.0.4/xcroco')
if XCROCO_PATH not in sys.path:
    sys.path.insert(0, XCROCO_PATH)

import inout as io      # I/O helpers to open CROCO grid + history/average files
import gridop as gop    # Grid operations: staggering, rotation, vertical interpolation, etc.
from model import Model # Model class describing the native CROCO grid conventions

print('Setup complete')

## 1. Parameters

In [ ]:
# ===== PATHS =====
DATA_DIR  = os.path.expanduser('./')
AVG_FILE  = os.path.join(DATA_DIR, 'croco_avg.0656.nc')   # CROCO time-averaged output file
GRID_FILE = os.path.join(DATA_DIR, 'grd_alb500sm.nc')     # CROCO grid file (curvilinear, ~500 m resolution)



print('Parameters defined')

## 2. Loading with xcroco

In [ ]:
# 'croco_native' tells xcroco to use the native CROCO grid conventions
# (staggered u/v/rho points, xi/eta index dimensions, etc.)
croco = Model('croco_native')

ds = io.open_files(
    croco,
    GRID_FILE,
    [AVG_FILE],
    grid_metrics=1,     # also load grid metrics (pm, pn, angle, ...)
    chunks={'t': 1},    # dask chunking: one time step per chunk
)

print(ds)

## 3. Time instant and fixed z-levels

In [ ]:
T_IDX = 0
ds1   = ds.isel(t=T_IDX)                              # select a single time snapshot
xgrid = gop.fast_xgcm_grid(ds1, croco, grid_metrics=1) # xgcm grid object (needed for staggered-grid ops)


# Real vertical coordinates (in meters) at rho points and w points,
# accounting for the free surface (z_sfc) — needed to interpolate
# variables from the native sigma/s-levels onto fixed z-levels.
z_rho = gop.get_z(croco, ds=ds1, z_sfc=ds1.z_sfc, xgrid=xgrid, vgrid='r', hgrid='r')
z_w   = gop.get_z(croco, ds=ds1, z_sfc=ds1.z_sfc, xgrid=xgrid, vgrid='w', hgrid='r')

print('z_rho:', z_rho.dims, z_rho.shape)
print('z_w:  ', z_w.dims,   z_w.shape)

# Non-uniform Z_LEVELS — real depths of the native sigma levels in the deepest column
h_vals = ds1.h.values                      # bathymetry, positive in meters
j_deep, i_deep = np.unravel_index(np.nanargmax(h_vals), h_vals.shape)

Z_LEVELS = np.sort(z_rho.values[:, j_deep, i_deep])   # bottom -> surface, native sigma levels

print(f'Domain: lat {float(ds1.lat.min()):.2f}-{float(ds1.lat.max()):.2f}  '
      f'lon {float(ds1.lon.min()):.2f}-{float(ds1.lon.max()):.2f}')
print(f'Bathymetry: min={h_vals.min():.1f}m  max={h_vals.max():.1f}m  (deepest at j={j_deep}, i={i_deep})')
print(f'Z_LEVELS: {len(Z_LEVELS)} levels, from {Z_LEVELS.min():.1f} to {Z_LEVELS.max():.1f} m')

dz = np.diff(Z_LEVELS)

fig, axes = plt.subplots(1, 3, figsize=(15, 6))

# 1. Bathymetry with reference column
ax = axes[0]
pc = ax.pcolormesh(ds1.lon.values, ds1.lat.values, h_vals, shading='auto', cmap='viridis')
ax.plot(ds1.lon.values[j_deep, i_deep], ds1.lat.values[j_deep, i_deep], 'r*', markersize=15)
fig.colorbar(pc, ax=ax, label="Depth (m)")
ax.set_title(f"Bathymetry — reference column\n(j={j_deep}, i={i_deep})")
ax.set_xlabel("Lon"); ax.set_ylabel("Lat")

# 2. Vertical distribution of levels
ax = axes[1]
ax.hlines(Z_LEVELS, 0, 1, color='#2471a3', linewidth=0.9)
ax.set_xticks([])
ax.set_ylabel("Depth (m)")
ax.set_title(f"Z_LEVELS ({len(Z_LEVELS)} levels)")

# 3. Vertical spacing between consecutive levels
ax = axes[2]
ax.plot(Z_LEVELS[:-1], dz, marker='o', markersize=3, color='#2471a3')
ax.set_xlabel("Depth (m)")
ax.set_ylabel("Spacing dz (m)")
ax.set_title("Vertical resolution")
ax.grid(alpha=0.3)

fig.suptitle(f"Z_LEVELS sigma-based — bathy max={h_vals.max():.1f} m")
fig.tight_layout()
plt.show()

print(f"Z_LEVELS: {len(Z_LEVELS)} levels, from {Z_LEVELS.min():.1f} to {Z_LEVELS.max():.1f} m")

In [ ]:
K = -2   # vertical index to plot (near-surface, since indexing starts from the bottom here)
lon = ds1.lon.values   # 2D curvilinear grid coordinates (native CROCO rho-grid, NOT a regular lat/lon grid)
lat = ds1.lat.values

fig, axs = plt.subplots(1, 3, figsize=(27, 5), constrained_layout=True)

for ax, data, title in zip(
    axs,
    [ds1.xcur.values[K, :, :-1],   # u lives on a staggered grid, 1 fewer point along x
     ds1.ycur.values[K, :-1, :],   # v lives on a staggered grid, 1 fewer point along y
     ds1.zcur.values[K, :,  :]],
    ['xcur u (m/s)', 'ycur v (m/s)', 'zcur w (m/s)']
):
    vmax = np.nanmax(np.abs(data))
    # w spans several orders of magnitude around zero -> symmetric log color scale;
    # u,v use a plain linear symmetric scale instead
    norm = SymLogNorm(linthresh=max(vmax*0.01, 1e-10), vmin=-vmax, vmax=vmax) if 'w' in title else None
    im = ax.pcolormesh(lon, lat, data, shading='auto',
                       cmap='RdBu_r', norm=norm,
                       vmin=-vmax if norm is None else None,
                       vmax= vmax if norm is None else None)
    fig.colorbar(im, ax=ax, label=title)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f'{title}  |  s={K}')

plt.suptitle(f'Raw croco_avg data — s={K}')
plt.show()

## 4. Geographic rotation + interpolation onto the rho grid

In [ ]:
# ============================================================
# FLAG: rotate velocities from grid-relative to geographic components
# ============================================================
# The native CROCO grid is rotated by a few degrees relative to true
# north (see ds1.angle). With ROTATE=True, u,v are rotated to true
# east/north components BEFORE being collocated onto rho points 

ROTATE = False  # set False to skip rotation (keeps grid-relative components)

if ROTATE:
    u_rot, v_rot = gop.rot_uv(ds1.xcur.compute(), ds1.ycur.compute(), ds1.angle, xgrid)
    u_geo = gop.to_rho(u_rot.compute(), xgrid)   # collocate onto rho points AFTER rotation
    v_geo = gop.to_rho(v_rot.compute(), xgrid)
else:
    u_geo = gop.to_rho(ds1.xcur.compute(), xgrid)
    v_geo = gop.to_rho(ds1.ycur.compute(), xgrid)

print('u_geo:', u_geo.dims, u_geo.shape)
print('v_geo:', v_geo.dims, v_geo.shape)
print('zcur: ', ds1.zcur.dims, ds1.zcur.shape)

## 5. Vertical interpolation onto fixed z-levels

In [ ]:
# ── u and v: isoslice (xgcm, linear interpolation) ──
u_zlevels = gop.isoslice(u_geo, Z_LEVELS, xgrid, target_data=z_rho.compute(), axis='z')
v_zlevels = gop.isoslice(v_geo, Z_LEVELS, xgrid, target_data=z_rho.compute(), axis='z')

# ── w: manual scipy interp1d (works around an xcroco bug on the z_w dimension) ──
z_w_np  = z_w.values        # (71, 860, 1700) — ordered bottom[0] -> surface[-1]
zcur_np = ds1.zcur.values   # (71, 860, 1700)
ny, nx  = z_w_np.shape[1], z_w_np.shape[2]
NZ      = len(Z_LEVELS)

w_np = np.full((NZ, ny, nx), np.nan)

# Column-by-column 1D interpolation, since isoslice can't be used directly
# for w here. Slow (double loop) but only run once at preprocessing time.
for j in range(ny):
    for i in range(nx):
        col_z = z_w_np[:, j, i]
        col_w = zcur_np[:, j, i]
        ok = np.isfinite(col_z) & np.isfinite(col_w)
        if ok.sum() < 2:
            continue  # fewer than 2 valid points: not enough to interpolate, skip this column
        f = interp1d(col_z[ok], col_w[ok], bounds_error=False, fill_value=np.nan)
        w_np[:, j, i] = f(Z_LEVELS)

w_zlevels = xr.DataArray(w_np, dims=u_zlevels.dims, coords=u_zlevels.coords)


# Drop leftover staggered-grid coordinates that no longer make sense
# once everything is collocated on the same (z, y, x) grid.
u_zlevels = u_zlevels.drop_vars(['lat_u', 'lon_u', 't'], errors='ignore')
v_zlevels = v_zlevels.drop_vars(['lat_v', 'lon_v', 't'], errors='ignore')
w_zlevels = w_zlevels.drop_vars(['lat_u', 'lon_u', 't'], errors='ignore')

print(f'u interpolated: {u_zlevels.shape},{type(u_zlevels)}')
print(f'v interpolated: {v_zlevels.shape},{type(v_zlevels)}')
print(f'w interpolated: {w_zlevels.shape},{type(w_zlevels)}')

In [ ]:
for name, da in [('u_zlevels', u_zlevels), ('v_zlevels', v_zlevels), ('w_zlevels', w_zlevels)]:
    print(f"{name}")
    print(f"  shape  : {da.shape}")
    print(f"  dims   : {da.dims}")
    print(f"  dtype  : {da.dtype}")
    print(f"  coords : {list(da.coords)}")
    print()

# Sanity check: u, v, w must all share the exact same z/lat/lon coordinates
# after interpolation, otherwise later code that assumes shared grids
# (masking, patch extraction, etc.) would silently misalign.
# u vs v: both come out of isoslice, so their coordinates are compared directly.
# w: w_zlevels is built in the cell above by COPYING u_zlevels' coords, so
# comparing its coordinates with u's would always pass. What can actually go
# wrong is the grid w was interpolated FROM: check that zcur/z_w live on the
# same horizontal (y, x) rho grid as u, and that u's z coordinate is Z_LEVELS
# (the levels w is interpolated onto with interp1d).
checks = {
    'u/v z_r'              : np.allclose(u_zlevels.coords['z_r'].values, v_zlevels.coords['z_r'].values),
    'u/v lat'              : np.allclose(u_zlevels.coords['lat'].values, v_zlevels.coords['lat'].values),
    'u/v lon'              : np.allclose(u_zlevels.coords['lon'].values, v_zlevels.coords['lon'].values),
    'u z_r == Z_LEVELS'    : np.allclose(u_zlevels.coords['z_r'].values, Z_LEVELS),
    'zcur on (y,x) of u'   : ds1.zcur.dims[-2:] == u_zlevels.dims[-2:] and ds1.zcur.shape[-2:] == u_zlevels.shape[-2:],
    'z_w  on (y,x) of u'   : z_w.dims[-2:] == u_zlevels.dims[-2:] and z_w.shape[-2:] == u_zlevels.shape[-2:],
}
for _name, _ok in checks.items():
    print(f"  {'✓' if _ok else '✗'} {_name}")
print('✓ all coordinates match' if all(checks.values()) else '✗ some coordinates differ')

## 6. Flip vertical axis — k=0 surface, k=-1 bottom

In [ ]:
# Flip the vertical axis so that k=0 is the surface and k=-1 is the bottom
# (isoslice/Z_LEVELS were built bottom-to-surface; this reorders everything
# to the more intuitive "k increases with depth" convention used from here on).
u_zlevels = u_zlevels.isel(z_r=slice(None, None, -1))
v_zlevels = v_zlevels.isel(z_r=slice(None, None, -1))
w_zlevels = w_zlevels.isel(z_r=slice(None, None, -1))
Z_LEVELS = Z_LEVELS[::-1]

print(f'Z_LEVELS[0]  = {Z_LEVELS[0]:.1f} m  <- surface')
print(f'Z_LEVELS[-1] = {Z_LEVELS[-1]:.1f} m  <- bottom')

## 7. Ibiza subdomain selection


In [ ]:
# Select the Ibiza subdomain by lat/lon range. Note: lat, lon are the
# NATIVE curvilinear grid coordinates (2D arrays), not a regular
# north-aligned lat/lon grid.

# ===== IBIZA SUBDOMAIN (lat/lon box used for training) =====
LAT_IB_MIN, LAT_IB_MAX = 36.90, 38.30   # widened 2026-09-02: was too small
LON_IB_MIN, LON_IB_MAX = 0.40, 2.00

mask_IB = (
    (lat >= LAT_IB_MIN) & (lat <= LAT_IB_MAX) &
    (lon >= LON_IB_MIN) & (lon <= LON_IB_MAX)
)

# Crop to a bounding box of indices to reduce array size before copying
j_idx, i_idx = np.where(mask_IB)
j_min, j_max = j_idx.min(), j_idx.max()
i_min, i_max = i_idx.min(), i_idx.max()

U_IB = u_zlevels.isel(y=slice(j_min, j_max+1), x=slice(i_min, i_max+1)).compute().values
V_IB = v_zlevels.isel(y=slice(j_min, j_max+1), x=slice(i_min, i_max+1)).compute().values
W_IB = w_zlevels.isel(y=slice(j_min, j_max+1), x=slice(i_min, i_max+1)).compute().values

lat_IB = lat[j_min:j_max+1, i_min:i_max+1]
lon_IB = lon[j_min:j_max+1, i_min:i_max+1]

# Mask out points that fall inside the index bounding box but outside the
# actual lat/lon range requested (the box is rectangular in INDEX space,
# not necessarily in lat/lon space on a rotated grid)
mask_2D_IB = ~mask_IB[j_min:j_max+1, i_min:i_max+1]
U_IB[:, mask_2D_IB] = np.nan
V_IB[:, mask_2D_IB] = np.nan
W_IB[:, mask_2D_IB] = np.nan

assert U_IB.shape == V_IB.shape == W_IB.shape, (
    f'Shape mismatch: U_IB={U_IB.shape}, V_IB={V_IB.shape}, W_IB={W_IB.shape}'
)
assert U_IB.shape[1:] == mask_2D_IB.shape, (
    f'mask_2D_IB shape {mask_2D_IB.shape} does not match spatial dims {U_IB.shape[1:]}'
)
print(f'✓ shapes OK: U_IB=V_IB=W_IB={U_IB.shape}, mask_2D_IB={mask_2D_IB.shape}')

print(f'Ibiza subdomain: {U_IB.shape}')
print(f'Valid points per level: {(~mask_2D_IB).sum()}')
print(f'lat: {lat_IB[~mask_2D_IB].min():.2f}-{lat_IB[~mask_2D_IB].max():.2f}')
print(f'lon: {lon_IB[~mask_2D_IB].min():.2f}-{lon_IB[~mask_2D_IB].max():.2f}')

## 7b. Validation zones — Alboran (reference) and Algerian (2nd zone)

Two independent out-of-sample zones, extracted the same way, right below. **Alboran** is the reference zone: Section 10 (FiLM CNN) tracks it during training and uses it only to decide WHEN to stop (no improvement for `R2_PATIENCE_EPOCHS_cnn` epochs, unless the learning rate reaches its minimum first); all reported results use the final model at the stopping epoch, and the best-on-Alboran checkpoint is kept only as a reference (Section 9, the per-level LR, has no model selection). **Algerian** is a second, fully independent zone (different current regime, no geographic overlap with Ibiza or Alboran, no land contamination), evaluated exactly the same way as an extra check, but not used to drive any model-selection decision.

In [ ]:
# ===== VALIDATION SET — Alboran =====
# Independent zone (different current regime from Ibiza) used to test
# generalization, not just in-sample fit.
LAT_NA_MIN, LAT_NA_MAX = 35.70, 36.40   # tightened 2026-09-02: kept as REFERENCE zone
LON_NA_MIN, LON_NA_MAX = -3.60, -2.20

mask_NA = (
    (lat >= LAT_NA_MIN) & (lat <= LAT_NA_MAX) &
    (lon >= LON_NA_MIN) & (lon <= LON_NA_MAX)
)

print(f'Valid Alboran points: {mask_NA.sum()}')

j_idx_na, i_idx_na = np.where(mask_NA)
j_min_na, j_max_na = j_idx_na.min(), j_idx_na.max()
i_min_na, i_max_na = i_idx_na.min(), i_idx_na.max()

U_NA = u_zlevels.isel(y=slice(j_min_na, j_max_na+1), x=slice(i_min_na, i_max_na+1)).compute().values
V_NA = v_zlevels.isel(y=slice(j_min_na, j_max_na+1), x=slice(i_min_na, i_max_na+1)).compute().values
W_NA = w_zlevels.isel(y=slice(j_min_na, j_max_na+1), x=slice(i_min_na, i_max_na+1)).compute().values

lat_NA = lat[j_min_na:j_max_na+1, i_min_na:i_max_na+1]
lon_NA = lon[j_min_na:j_max_na+1, i_min_na:i_max_na+1]

mask_2D_NA = ~mask_NA[j_min_na:j_max_na+1, i_min_na:i_max_na+1]

U_NA[:, mask_2D_NA] = np.nan
V_NA[:, mask_2D_NA] = np.nan
W_NA[:, mask_2D_NA] = np.nan

assert U_NA.shape == V_NA.shape == W_NA.shape, (
    f'Shape mismatch: U_NA={U_NA.shape}, V_NA={V_NA.shape}, W_NA={W_NA.shape}'
)
assert U_NA.shape[1:] == mask_2D_NA.shape, (
    f'mask_2D_NA shape {mask_2D_NA.shape} does not match spatial dims {U_NA.shape[1:]}'
)
print(f'✓ shapes OK: U_NA=V_NA=W_NA={U_NA.shape}, mask_2D_NA={mask_2D_NA.shape}')

print(f'lat: {lat_NA[~mask_2D_NA].min():.2f} - {lat_NA[~mask_2D_NA].max():.2f}')
print(f'lon: {lon_NA[~mask_2D_NA].min():.2f} - {lon_NA[~mask_2D_NA].max():.2f}')

In [ ]:
# ===== VALIDATION SET — Algerian (2nd independent zone) =====
# Same treatment as the Alboran cell above: independent zone, different
# current regime, used purely as an extra out-of-sample check.
LAT_AG_MIN, LAT_AG_MAX = 36.40, 37.40
LON_AG_MIN, LON_AG_MAX = -1.50, -0.20

mask_AG = (
    (lat >= LAT_AG_MIN) & (lat <= LAT_AG_MAX) &
    (lon >= LON_AG_MIN) & (lon <= LON_AG_MAX)
)

print(f'Valid Algerian points: {mask_AG.sum()}')

j_idx_ag, i_idx_ag = np.where(mask_AG)
j_min_ag, j_max_ag = j_idx_ag.min(), j_idx_ag.max()
i_min_ag, i_max_ag = i_idx_ag.min(), i_idx_ag.max()

U_AG = u_zlevels.isel(y=slice(j_min_ag, j_max_ag+1), x=slice(i_min_ag, i_max_ag+1)).compute().values
V_AG = v_zlevels.isel(y=slice(j_min_ag, j_max_ag+1), x=slice(i_min_ag, i_max_ag+1)).compute().values
W_AG = w_zlevels.isel(y=slice(j_min_ag, j_max_ag+1), x=slice(i_min_ag, i_max_ag+1)).compute().values

lat_AG = lat[j_min_ag:j_max_ag+1, i_min_ag:i_max_ag+1]
lon_AG = lon[j_min_ag:j_max_ag+1, i_min_ag:i_max_ag+1]

mask_2D_AG = ~mask_AG[j_min_ag:j_max_ag+1, i_min_ag:i_max_ag+1]

U_AG[:, mask_2D_AG] = np.nan
V_AG[:, mask_2D_AG] = np.nan
W_AG[:, mask_2D_AG] = np.nan

assert U_AG.shape == V_AG.shape == W_AG.shape, (
    f'Shape mismatch: U_AG={U_AG.shape}, V_AG={V_AG.shape}, W_AG={W_AG.shape}'
)
assert U_AG.shape[1:] == mask_2D_AG.shape, (
    f'mask_2D_AG shape {mask_2D_AG.shape} does not match spatial dims {U_AG.shape[1:]}'
)
print(f'\u2713 shapes OK: U_AG=V_AG=W_AG={U_AG.shape}, mask_2D_AG={mask_2D_AG.shape}')

print(f'lat: {lat_AG[~mask_2D_AG].min():.2f} - {lat_AG[~mask_2D_AG].max():.2f}')
print(f'lon: {lon_AG[~mask_2D_AG].min():.2f} - {lon_AG[~mask_2D_AG].max():.2f}')


## 10. FiLM CNN — setup (no training)

Same setup cell as `notebook_nuredduna_seeds.ipynb` (parameters, model class, patch extraction, standardization from Ibiza). It is needed to rebuild the model and to standardize the inputs exactly as in training; nothing is trained in this notebook.

In [ ]:
# ============================================================
# FiLM ENSEMBLE — SETUP (run once): parameters, model/helpers, patch
# extraction and standardization. Identical to the FiLM cell of
# notebook_nuredduna.ipynb; the seed-dependent part (training +
# evaluation) is in run_seed() below.
# ============================================================
import torch
import torch.nn as nn
import numpy as np
import time
import psutil
import os
import sys
import json
import csv
import itertools
from datetime import datetime
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.metrics import r2_score
from matplotlib.colors import SymLogNorm
# ============================================================
# Requires already loaded upstream (from the preprocessing block):
#   U_IB, V_IB, W_IB   -> (nz, ny, nx)  Ibiza fields (training)
#   U_NA, V_NA, W_NA   -> (nz, ny_na, nx_na)  Alboran fields (validation, REFERENCE)
#   lat_NA, lon_NA     -> 2D Alboran grids for plotting
#   U_AG, V_AG, W_AG   -> (nz, ny_ag, nx_ag)  Algerian fields (2nd validation)
#   lat_AG, lon_AG     -> 2D Algerian grids for plotting
#   Z_LEVELS           -> real depths (meters) per level
# ============================================================
import random
import subprocess

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ============================================================
# PARAMETERS — edit only here
# ============================================================
r_cnn          = 3         # HORIZONTAL stencil radius: patch = (2r+1) x (2r+1)
K_START_cnn    = 2         # level (index) of u,v used as input — e.g. surface/near-surface
N_TARGET_cnn   = 16        # levels 2..17 -> z down to about -116.6 m, stays above -150 m
K_PRED_cnn     = [2,7,12,17]   # absolute levels used for metrics/plots on Alboran and Algerian
ROW_BATCH_cnn   = 2048      # target number of (patch, level) ROWS per training batch: batches are
                            # built from whole patches, PATCH_BATCH_cnn = ROW_BATCH_cnn // N_TARGET_cnn
                            # patches with all their levels (conv1 once per patch, see forward_grouped)
PRED_BATCH_cnn  = 512       # batching for validation/prediction
K_EXTRAPOLATE_cnn = 20        # modest extrapolation just past training range (was 35)
N_EXTRAP_FIT_cnn  = 4         # deepest training levels used for the least-squares fit of mean_w(z)
                              # (linear) and log std_w(z) (linear) when predicting BELOW the training range
VALIDATE_EVERY_EPOCHS = 25      # cadence once training has settled
# Denser validation at the start, to resolve how val R2 rises: list of
# (up to epoch, check every N epochs); after the last entry the cadence
# is VALIDATE_EVERY_EPOCHS.
VALIDATE_SCHEDULE = [(25, 1)]

# Regularization knobs -- grouped here too (not just architecture/training
# params) so config.json is fully self-describing: no need to go read the
# model class / optimizer code to know how a run was
# regularized.
DROPOUT2D_P         = 0.1   # spatial dropout on conv feature maps (history: 0.1 -> 0.2 -> 0.1)
FC_DROPOUT_P        = 0.2   # dropout in the FC head (history: 0.3 -> 0.2)
WD_OTHER            = 1e-3  # weight_decay on conv1/conv2/FC weights (history: 1e-4 -> 1e-3 -> 3e-3 -> 1e-3)
WD_FILM             = 5e-5  # weight_decay (L2) on the FiLM generator only, see optimizer comment (history: 0.0 -> 1e-5 -> 5e-5 -> 1e-5 -> 5e-5)
# Stopping: NO fixed number of epochs. Training stops at the FIRST of
#  (a) the lr reaches LR_MIN_cnn: ReduceLROnPlateau (on the TRAIN loss) has
#      exhausted its reductions -> training has converged;
#  (b) the mean val R² on Alboran (K_PRED_cnn levels) has not improved for
#      R2_PATIENCE_EPOCHS_cnn epochs.
# Alboran only decides WHEN to stop: all evaluation uses the model at the
# stopping epoch (the final model), not the best-on-Alboran checkpoint.
LR_MIN_cnn             = 1e-6  # min lr of ReduceLROnPlateau = convergence criterion (a)
R2_PATIENCE_EPOCHS_cnn = 300   # criterion (b), in epochs (was 250 = 10 checks x 25)

# Save the run config alongside the results, so a results.npz/log.txt
# found later is self-describing without needing the notebook.
_config = dict(r_cnn=r_cnn, K_START_cnn=K_START_cnn, N_TARGET_cnn=N_TARGET_cnn,
                K_PRED_cnn=K_PRED_cnn,
                ROW_BATCH_cnn=ROW_BATCH_cnn, PRED_BATCH_cnn=PRED_BATCH_cnn,
                K_EXTRAPOLATE_cnn=K_EXTRAPOLATE_cnn, N_EXTRAP_FIT_cnn=N_EXTRAP_FIT_cnn,
                VALIDATE_EVERY_EPOCHS=VALIDATE_EVERY_EPOCHS,
                validate_schedule=VALIDATE_SCHEDULE,
                dropout2d_p=DROPOUT2D_P, fc_dropout_p=FC_DROPOUT_P,
                weight_decay_other=WD_OTHER, weight_decay_film=WD_FILM,
                lr_min=LR_MIN_cnn, r2_patience_epochs=R2_PATIENCE_EPOCHS_cnn,
                evaluated_model='final', device=str(device))
# (_config is written to config.json in every seed folder, together with the seed)


def extract_patches_with_all_levels(u_field, v_field, W_levels_full, r):
    """Extracts all (2r+1)x(2r+1) patches whose u,v neighbourhood is fully
    valid, together with the w vector at ALL target levels (even if some
    levels are NaN — they get filtered afterwards, row by row).
    Returns: patches (n_patches,2,ss,ss), y_all_levels (n_patches,n_levels)
             (can contain NaN), centers_j, centers_i (n_patches,) — indices
             in the original field.
    """
    ny, nx = u_field.shape
    ss = 2 * r + 1
    u_win = sliding_window_view(u_field, (ss, ss))
    v_win = sliding_window_view(v_field, (ss, ss))
    patch_valid = ~np.isnan(u_win).any(axis=(2, 3)) & ~np.isnan(v_win).any(axis=(2, 3))
    jj, ii = np.where(patch_valid)
    patches = np.stack([u_win[jj, ii], v_win[jj, ii]], axis=1)
    centers_j, centers_i = jj + r, ii + r
    y_all_levels = W_levels_full[:, centers_j, centers_i].T
    return patches, y_all_levels, centers_j, centers_i


def build_flat_training_rows(y_all_levels, z_of_level):
    """Expands (n_patches, n_levels) into independent (patch_idx, z, y) rows,
    dropping ONLY the (patch,level) combinations where w is NaN."""
    valid = ~np.isnan(y_all_levels)
    patch_idx, level_idx = np.where(valid)
    y_flat = y_all_levels[patch_idx, level_idx].astype(np.float32)
    z_flat = z_of_level[level_idx].astype(np.float32)
    return patch_idx, level_idx, z_flat, y_flat


class FiLM(nn.Module):
    """Generates gamma, beta from z and modulates a (B,C,H,W) tensor channel-wise:
    output = gamma * x + beta, applied identically over the whole spatial window."""
    def __init__(self, z_dim, num_channels):
        super().__init__()
        self.to_gamma_beta = nn.Linear(z_dim, num_channels * 2)
        # Canonical FiLM init: gamma starts at 1, beta at 0, i.e. FiLM starts
        # as the identity. This is only the INITIAL CONDITION: gamma/beta are
        # then free to move away from it during training. Without it,
        # nn.Linear's default weights give gamma~0 at the start and conv1's
        # features would be almost zeroed out before the network has learned
        # anything.
        nn.init.zeros_(self.to_gamma_beta.weight)
        nn.init.zeros_(self.to_gamma_beta.bias)
        with torch.no_grad():
            self.to_gamma_beta.bias[:num_channels].fill_(1.0)  # half of gamma -> 1, beta stays 0

    def forward(self, x, z):
        gamma_beta = self.to_gamma_beta(z)
        gamma, beta = gamma_beta.chunk(2, dim=1)
        gamma = gamma.unsqueeze(-1).unsqueeze(-1)
        beta  = beta.unsqueeze(-1).unsqueeze(-1)
        return gamma * x + beta


class ZCondStencilCNN_FiLM(nn.Module):
    """CNN conditioned on z via a single FiLM layer, applied BEFORE
    conv2 (it modulates conv1's output). conv1 sees only the raw (u,v)
    stencil (no z at all) and always extracts the same depth-invariant
    local features; FiLM then rescales/shifts those 8 feature maps as a
    function of z, and conv2 -- a fixed, non-z-dependent 8->16
    recombination -- operates on that already depth-conditioned input.
    Single entry point for z, ~7.8k total parameters."""
    def __init__(self, stencil_size, z_dim=1):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 8, kernel_size=3, padding=1)   # padding=1 -- same-size output (7x7 -> 7x7)
        self.film1 = FiLM(z_dim, 8)                               # only entry point for z -- applied BEFORE conv2
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=0)   # no FiLM: fixed recombination of the z-modulated input
        self.relu = nn.ReLU()
        self.spatial_dropout = nn.Dropout2d(DROPOUT2D_P)  # value set in the PARAMETERS block above
        flat_dim = 16 * (stencil_size - 2) * (stencil_size - 2)   # conv1 is now 'same' padding (no shrink); only conv2 is valid (-2)
        self.fc = nn.Sequential(
            nn.Linear(flat_dim, 16), nn.ReLU(), nn.Dropout(FC_DROPOUT_P),
            nn.Linear(16, 1)
        )

    def features(self, x_img):
        """Depth-invariant part (conv1 + ReLU): depends only on the (u,v) stencil."""
        return self.relu(self.conv1(x_img))

    def head(self, h1, z):
        """z-dependent part. Dropout on conv1's output is applied HERE, i.e.
        per (patch, z) row, so every row keeps its own independent mask
        exactly as before the conv1 hoisting."""
        x = self.spatial_dropout(h1)
        x = self.film1(x, z)                                      # z enters here, BEFORE conv2
        x = self.spatial_dropout(self.relu(self.conv2(x)))        # fixed recombination of the z-modulated input
        feat = x.flatten(start_dim=1)
        return self.fc(feat).squeeze(-1)

    def forward(self, x_img, z):
        return self.head(self.features(x_img), z)

    def forward_grouped(self, x_unique, inv, z):
        """Same as forward(x_unique[inv], z), but conv1 runs once per distinct
        patch; its output is then copied to every row that shares the patch."""
        return self.head(self.features(x_unique)[inv], z)


def predict_zcond_in_batches(model, patches_tensor, patch_idx, z_flat_s, device,
                              batch_size=PRED_BATCH_cnn):
    """Mini-batch inference. Within each chunk, conv1 is computed once per
    distinct patch (forward_grouped) and shared by all rows using it."""
    model.eval()
    n = len(patch_idx)
    z_t = torch.tensor(z_flat_s, dtype=torch.float32).unsqueeze(1)
    out = []
    with torch.no_grad():
        for start in range(0, n, batch_size):
            end = start + batch_size
            uniq_b, inv_b = np.unique(patch_idx[start:end], return_inverse=True)
            xb = patches_tensor[uniq_b].to(device)
            inv_t = torch.as_tensor(inv_b.ravel(), dtype=torch.long, device=device)
            zb = z_t[start:end].to(device)
            out.append(model.forward_grouped(xb, inv_t, zb).cpu())
    return torch.cat(out, dim=0).numpy()


# ------------------------------------------------------------
# 1. Extraction — Ibiza (training): horizontal patches + all w levels
# ------------------------------------------------------------
u_cnn = U_IB[K_START_cnn]
v_cnn = V_IB[K_START_cnn]
W_IB_slice = W_IB[K_START_cnn : K_START_cnn + N_TARGET_cnn]
z_levels_ib = Z_LEVELS[K_START_cnn : K_START_cnn + N_TARGET_cnn]
patches_ib, y_all_ib, _, _ = extract_patches_with_all_levels(u_cnn, v_cnn, W_IB_slice, r_cnn)
patch_idx_ib, level_idx_ib, z_flat_ib, y_flat_ib = build_flat_training_rows(y_all_ib, z_levels_ib)
stencil_size_cnn = 2 * r_cnn + 1
print(f'Ibiza — stencil {stencil_size_cnn}×{stencil_size_cnn} | valid patches: {len(patches_ib)} | '
      f'rows (patch,level) for training: {len(y_flat_ib)}')
if len(y_flat_ib) == 0:
    raise RuntimeError('No valid training rows — check r_cnn/K_START_cnn')

# ------------------------------------------------------------
# 2. Standardization
# ------------------------------------------------------------
mean_u_cnn, std_u_cnn = patches_ib[:, 0].mean(), patches_ib[:, 0].std()
mean_v_cnn, std_v_cnn = patches_ib[:, 1].mean(), patches_ib[:, 1].std()
patches_ib_s = patches_ib.copy().astype(np.float32)
patches_ib_s[:, 0] = (patches_ib[:, 0] - mean_u_cnn) / std_u_cnn
patches_ib_s[:, 1] = (patches_ib[:, 1] - mean_v_cnn) / std_v_cnn
mean_z_cnn, std_z_cnn = z_levels_ib.mean(), z_levels_ib.std()
z_flat_ib_s = (z_flat_ib - mean_z_cnn) / std_z_cnn
mean_w_per_level = np.array([np.nanmean(y_all_ib[:, t]) for t in range(N_TARGET_cnn)])
std_w_per_level  = np.array([np.nanstd(y_all_ib[:, t])  for t in range(N_TARGET_cnn)]) + 1e-10
y_flat_ib_s = (y_flat_ib - mean_w_per_level[level_idx_ib]) / std_w_per_level[level_idx_ib]
patches_ib_tensor = torch.tensor(patches_ib_s, dtype=torch.float32)

# Target stats (mean_w, std_w) at an ARBITRARY depth z [m]. They are only
# defined on the training levels, so:
#  - z between two training levels -> linear interpolation in z between the
#    two nearest levels (weights by distance); exact value on a level;
#  - z below the deepest training level -> least-squares fit on the
#    N_EXTRAP_FIT_cnn deepest levels, evaluated at z: mean_w(z) linear,
#    log(std_w)(z) linear and then std_w = exp(...), so std_w stays > 0;
#  - z above the shallowest training level -> not supported (error).
_z_train = np.asarray(z_levels_ib, dtype=np.float64)
_asc = np.argsort(_z_train)                     # ascending z: deepest first

def w_stats_at_z(z):
    z = float(z)
    z_top, z_bot = _z_train.max(), _z_train.min()
    if z > z_top + 1e-6:
        raise ValueError(f'z={z:.2f} m is above the shallowest training level ({z_top:.2f} m): not supported')
    if z >= z_bot - 1e-6:                        # inside the training range
        m = np.interp(z, _z_train[_asc], mean_w_per_level[_asc])
        s = np.interp(z, _z_train[_asc], std_w_per_level[_asc])
    else:                                        # below the training range
        sel = _asc[:N_EXTRAP_FIT_cnn]            # the N deepest training levels
        m = np.polyval(np.polyfit(_z_train[sel], mean_w_per_level[sel], 1), z)
        # std_w: linear fit of log(std_w) in z, then back to std_w with exp (always > 0)
        s = np.exp(np.polyval(np.polyfit(_z_train[sel], np.log(std_w_per_level[sel]), 1), z))
    return float(m), float(s)

# ------------------------------------------------------------
# 3. Extraction — Alboran (validation)
# ------------------------------------------------------------
u_na, v_na = U_NA[K_START_cnn], V_NA[K_START_cnn]
ny_na, nx_na = u_na.shape
W_NA_slice = W_NA[K_START_cnn : K_START_cnn + N_TARGET_cnn]
z_levels_na = Z_LEVELS[K_START_cnn : K_START_cnn + N_TARGET_cnn]
patches_na, y_all_na, centers_j_na, centers_i_na = extract_patches_with_all_levels(u_na, v_na, W_NA_slice, r_cnn)
patches_na_s = patches_na.copy().astype(np.float32)
patches_na_s[:, 0] = (patches_na[:, 0] - mean_u_cnn) / std_u_cnn
patches_na_s[:, 1] = (patches_na[:, 1] - mean_v_cnn) / std_v_cnn
patches_na_tensor = torch.tensor(patches_na_s, dtype=torch.float32)
print(f'Alboran — valid patches: {len(patches_na)}')

# ------------------------------------------------------------
# 3b. Extraction — Algerian (2nd independent validation zone)
# ------------------------------------------------------------
u_ag, v_ag = U_AG[K_START_cnn], V_AG[K_START_cnn]
ny_ag, nx_ag = u_ag.shape
W_AG_slice = W_AG[K_START_cnn : K_START_cnn + N_TARGET_cnn]
patches_ag, y_all_ag, centers_j_ag, centers_i_ag = extract_patches_with_all_levels(u_ag, v_ag, W_AG_slice, r_cnn)
patches_ag_s = patches_ag.copy().astype(np.float32)
patches_ag_s[:, 0] = (patches_ag[:, 0] - mean_u_cnn) / std_u_cnn
patches_ag_s[:, 1] = (patches_ag[:, 1] - mean_v_cnn) / std_v_cnn
patches_ag_tensor = torch.tensor(patches_ag_s, dtype=torch.float32)
print(f'Algerian — valid patches: {len(patches_ag)}')



## 12. Input-noise robustness on Algerian

Question: how much do the per-level LR and the FiLM CNN, both trained on clean Ibiza data, still recover the **true** $w$ of Algerian when their $(u,v)$ input contains an error.

Error added to the input level `K_START_cnn` **before** extracting the stencils (white Gaussian, independent pixel by pixel and between $u$ and $v$):

$$u' = u + \alpha\,\sigma_u\,\mathcal{N}(0,1), \qquad v' = v + \alpha\,\sigma_v\,\mathcal{N}(0,1), \qquad \alpha = 1\%,\ 5\%,\ 10\%$$

with $\sigma_u,\sigma_v$ the spatial std of $u, v$ over the Algerian sea pixels (printed in m/s). The target is always the true $w$ (no noise on $w$). As in He & Mahadevan (2024) the noise is white and only on the velocities; unlike them, nothing is retrained (models trained on clean data, applied to noisy input).

- **Models**: per-level LR refitted on Ibiza (deterministic, identical to Section 9); FiLM = `model_final.pt` of the first 50 trained seeds in `cnn_seeds_film/`.
- **Noise realizations**: one per seed (rng seed `NOISE_RNG_OFFSET + seed`), shared by the three $\alpha$ (only the amplitude changes) and by LR and FiLM (same perturbed input). The LR therefore also gets a mean ± std.
- **LR expected**: for white noise the LR prediction changes by $\sum_i\beta_i\varepsilon_i$, so the expected R² is $R^2_{clean} - \alpha^2(\sigma_u^2\sum\beta_u^2+\sigma_v^2\sum\beta_v^2)/\mathrm{Var}(w)$: a check of the simulation.
- **Checks**: clean LR R² vs `lr_reference.json`, clean FiLM R² vs each seed's `result.json`.
- **Output** in `cnn_seeds_film/noise_algerian/`: `config.json`, `seed_XXX.csv` (resumable: done seeds are skipped), `noise_r2_summary.csv` (mean, std, paired drop ΔR² per model/α/level), `noise_r2_all_seeds.csv`, `noise_r2_vs_alpha.png/pdf`, `noise_r2_vs_depth.png/pdf`.

In [ ]:
# ============================================================
# 12. NOISE ROBUSTNESS — Algerian only, error on the u,v INPUTS
#     Requires the FiLM ENSEMBLE SETUP cell above (model class, helpers,
#     patches, standardization). No training: the per-level LR is refitted
#     on Ibiza (deterministic, identical to Section 9) and the FiLM uses the
#     final weights (model_final.pt) of the already trained seeds.
#
#     Input error (white Gaussian, independent pixel by pixel and between
#     u and v), added to the input level K_START_cnn BEFORE extracting the
#     stencils:
#         u' = u + alpha * sigma_u * N(0,1)
#         v' = v + alpha * sigma_v * N(0,1)
#     sigma_u, sigma_v = spatial std of u, v over the Algerian sea pixels.
#     The target is ALWAYS the true w of Algerian (no noise on w).
#     One noise realization per seed (rng seed = NOISE_RNG_OFFSET + seed),
#     shared by the three alphas (only the amplitude changes) and by LR
#     and FiLM (both see exactly the same perturbed input).
# ============================================================
import os, json, csv, time
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# ============================================================
# PARAMETERS — edit only here
# ============================================================
NOISE_ALPHAS     = [0.01, 0.05, 0.10]       # noise std as a fraction of sigma_u, sigma_v
NOISE_SEEDS      = list(range(50))          # trained FiLM seeds used (seed_000 ... seed_049)
ENSEMBLE_DIR     = './cnn_seeds_film'       # where the trained seeds are
NOISE_DIR        = os.path.join(ENSEMBLE_DIR, 'noise_algerian')
NOISE_RNG_OFFSET = 100_000                  # noise rng independent of the training seeds
PRED_BATCH_NOISE = 16 * PRED_BATCH_cnn      # rows per inference batch (no training here)
# ============================================================

assert r_cnn == 3 and K_START_cnn == 2 and N_TARGET_cnn == 16, \
    'the LR refit below reproduces Section 9 only for r=3, K_START=2, N_TARGET=16'
os.makedirs(NOISE_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. Clean Algerian input level and noise amplitude
# ------------------------------------------------------------
u_ag0 = np.asarray(U_AG[K_START_cnn], dtype=np.float64)
v_ag0 = np.asarray(V_AG[K_START_cnn], dtype=np.float64)
_sea = ~np.isnan(u_ag0) & ~np.isnan(v_ag0)
sigma_u_ag = float(np.std(u_ag0[_sea]))
sigma_v_ag = float(np.std(v_ag0[_sea]))
print(f'Algerian input level k={K_START_cnn} (z={Z_LEVELS[K_START_cnn]:.1f} m), {_sea.sum()} sea pixels')
print(f'sigma_u = {sigma_u_ag:.4f} m/s | sigma_v = {sigma_v_ag:.4f} m/s')
for a in NOISE_ALPHAS:
    print(f'  alpha = {100*a:>4.0f}%  ->  noise std: u {100*a*sigma_u_ag:.3f} cm/s, v {100*a*sigma_v_ag:.3f} cm/s')

_cfg = dict(alphas=NOISE_ALPHAS, rng_offset=NOISE_RNG_OFFSET,
            sigma_u_m_s=sigma_u_ag, sigma_v_m_s=sigma_v_ag,
            input_level=K_START_cnn, stencil_r=r_cnn,
            definition="u' = u + alpha*sigma_u*N(0,1), v' = v + alpha*sigma_v*N(0,1); "
                       "white Gaussian, independent per pixel and between u and v; "
                       "sigma = spatial std over the Algerian sea pixels; target = true w; "
                       "one realization per seed, shared by the alphas and by LR/FiLM")
_cfg_path = os.path.join(NOISE_DIR, 'config.json')
if os.path.exists(_cfg_path):
    _old = json.load(open(_cfg_path))
    _same = (_old['alphas'] == _cfg['alphas'] and _old['rng_offset'] == _cfg['rng_offset']
             and abs(_old['sigma_u_m_s'] - sigma_u_ag) < 1e-9 and abs(_old['sigma_v_m_s'] - sigma_v_ag) < 1e-9)
    if not _same:
        raise RuntimeError(f'{_cfg_path} has a different configuration: use another NOISE_DIR '
                           f'(the per-seed files already there would be mixed with the new ones)')
else:
    json.dump(_cfg, open(_cfg_path, 'w'), indent=2)

# ------------------------------------------------------------
# 2. Per-level LR refitted on Ibiza (identical to Section 9: same 7x7
#    stencil, features [u.ravel(), v.ravel()], one model per level)
# ------------------------------------------------------------
def lr_features(patches):
    """(n, 2, ss, ss) raw patches -> (n, 2*ss*ss) = [u.ravel(), v.ravel()] as in Section 9."""
    n = len(patches)
    return np.concatenate([patches[:, 0].reshape(n, -1), patches[:, 1].reshape(n, -1)], axis=1)

_X_ib_lr = lr_features(patches_ib)
lr_models = {}
for t in range(N_TARGET_cnn):
    _ok = ~np.isnan(y_all_ib[:, t])
    lr_models[t] = LinearRegression().fit(_X_ib_lr[_ok], y_all_ib[_ok, t])
print(f'LR refitted on Ibiza: {len(lr_models)} levels, {_X_ib_lr.shape[1]} features')

# ------------------------------------------------------------
# 3. Evaluation helpers (Algerian, all 16 levels)
# ------------------------------------------------------------
z_levels_ag = Z_LEVELS[K_START_cnn : K_START_cnn + N_TARGET_cnn]
K_ALL = list(range(K_START_cnn, K_START_cnn + N_TARGET_cnn))
# (patch, level) rows of Algerian with a valid true w: patch-major order, so
# conv1 is computed once per patch inside each inference batch
_pidx_ag, _lidx_ag, _zflat_ag, _yflat_ag = build_flat_training_rows(y_all_ag, z_levels_ag)
_zflat_ag_s = (_zflat_ag - mean_z_cnn) / std_z_cnn
_var_true_ag = {t: float(np.var(y_all_ag[~np.isnan(y_all_ag[:, t]), t])) for t in range(N_TARGET_cnn)}


def algerian_patches(u_field, v_field):
    """Raw (n,2,7,7) Algerian patches of a (possibly perturbed) input; the
    noise does not change the NaN pattern, so the pixels are the clean ones."""
    p, _, cj, ci = extract_patches_with_all_levels(u_field, v_field, W_AG_slice, r_cnn)
    assert np.array_equal(cj, centers_j_ag) and np.array_equal(ci, centers_i_ag)
    return p


def r2_lr(patches):
    X = lr_features(patches)
    out = {}
    for t in range(N_TARGET_cnn):
        ok = ~np.isnan(y_all_ag[:, t])
        out[t] = float(r2_score(y_all_ag[ok, t], lr_models[t].predict(X[ok])))
    return out


def r2_film(model, patches):
    ps = patches.astype(np.float32).copy()
    ps[:, 0] = (patches[:, 0] - mean_u_cnn) / std_u_cnn
    ps[:, 1] = (patches[:, 1] - mean_v_cnn) / std_v_cnn
    y_s = predict_zcond_in_batches(model, torch.tensor(ps), _pidx_ag, _zflat_ag_s, device,
                                   batch_size=PRED_BATCH_NOISE)
    y = y_s * std_w_per_level[_lidx_ag] + mean_w_per_level[_lidx_ag]
    return {t: float(r2_score(_yflat_ag[_lidx_ag == t], y[_lidx_ag == t])) for t in range(N_TARGET_cnn)}


# ------------------------------------------------------------
# 4. Clean reference (alpha = 0) and LR check against Section 9 values
# ------------------------------------------------------------
_patches_clean = algerian_patches(u_ag0, v_ag0)
r2_lr_clean = r2_lr(_patches_clean)
_lr_ref_path = os.path.join(ENSEMBLE_DIR, 'lr_reference.json')
if os.path.exists(_lr_ref_path):
    _ref = json.load(open(_lr_ref_path)).get('r2', {}).get('Algerian', {})
    for k in K_PRED_cnn:
        if str(k) in _ref:
            print(f'LR clean Algerian k={k}: {r2_lr_clean[k - K_START_cnn]:.4f} '
                  f'(lr_reference.json: {_ref[str(k)]["r2"]:.4f})')

# Expected LR R2 with noise (exact in expectation, white noise): the LR
# prediction changes by sum_i beta_i eps_i, so the MSE grows by
# alpha^2 (sigma_u^2 sum beta_u^2 + sigma_v^2 sum beta_v^2).
_nb = (2 * r_cnn + 1) ** 2
lr_expected = {}
for a in NOISE_ALPHAS:
    for t in range(N_TARGET_cnn):
        c = lr_models[t].coef_
        d_mse = a**2 * (sigma_u_ag**2 * np.sum(c[:_nb]**2) + sigma_v_ag**2 * np.sum(c[_nb:]**2))
        lr_expected[(a, t)] = r2_lr_clean[t] - d_mse / _var_true_ag[t]

# ------------------------------------------------------------
# 5. Loop over seeds (resumable: one CSV per seed in NOISE_DIR)
# ------------------------------------------------------------
_FIELDS = ['model', 'seed', 'alpha', 'k', 'z_m', 'r2']
_t_all = time.time()
for seed in NOISE_SEEDS:
    _out = os.path.join(NOISE_DIR, f'seed_{seed:03d}.csv')
    if os.path.exists(_out):
        print(f'seed {seed:03d}: already done, skipped')
        continue
    _wpath = os.path.join(ENSEMBLE_DIR, f'seed_{seed:03d}', 'model_final.pt')
    if not os.path.exists(_wpath):
        print(f'seed {seed:03d}: {_wpath} not found, skipped')
        continue
    _t0 = time.time()
    model_n = ZCondStencilCNN_FiLM(stencil_size_cnn).to(device)
    model_n.load_state_dict(torch.load(_wpath, map_location=device))
    model_n.eval()

    rows = []
    _r2_film_clean = r2_film(model_n, _patches_clean)
    # check against the value stored by run_seed for this seed
    _res_path = os.path.join(ENSEMBLE_DIR, f'seed_{seed:03d}', 'result.json')
    if os.path.exists(_res_path):
        _rf = json.load(open(_res_path))['r2_final']
        _dmax = max(abs(_r2_film_clean[k - K_START_cnn] - _rf[f'Algerian|{k}']) for k in K_PRED_cnn)
        if _dmax > 1e-3:
            print(f'  WARNING seed {seed:03d}: clean R2 differs from result.json by {_dmax:.2e}')
    for t, k in enumerate(K_ALL):
        rows.append(['FiLM', seed, 0.0, k, float(Z_LEVELS[k]), _r2_film_clean[t]])
        rows.append(['LR',   seed, 0.0, k, float(Z_LEVELS[k]), r2_lr_clean[t]])

    _rng = np.random.default_rng(NOISE_RNG_OFFSET + seed)
    _eps_u = _rng.standard_normal(u_ag0.shape)
    _eps_v = _rng.standard_normal(v_ag0.shape)
    for a in NOISE_ALPHAS:
        _p = algerian_patches(u_ag0 + a * sigma_u_ag * _eps_u,     # NaN (land/outside) stays NaN
                              v_ag0 + a * sigma_v_ag * _eps_v)
        _rf_n, _rl_n = r2_film(model_n, _p), r2_lr(_p)
        for t, k in enumerate(K_ALL):
            rows.append(['FiLM', seed, a, k, float(Z_LEVELS[k]), _rf_n[t]])
            rows.append(['LR',   seed, a, k, float(Z_LEVELS[k]), _rl_n[t]])

    with open(_out + '.tmp', 'w', newline='') as f:
        w = csv.writer(f); w.writerow(_FIELDS); w.writerows(rows)
    os.replace(_out + '.tmp', _out)                 # file appears only when complete
    _k2 = K_PRED_cnn[0] - K_START_cnn
    print(f'seed {seed:03d}: {time.time() - _t0:5.1f} s | FiLM k={K_PRED_cnn[0]} R2 clean {_r2_film_clean[_k2]:.3f} -> '
          + ' '.join(f'{100*a:.0f}%: {r:.3f}' for a, r in
                     zip(NOISE_ALPHAS, [x[5] for x in rows if x[0] == 'FiLM' and x[2] > 0 and x[3] == K_PRED_cnn[0]])),
          flush=True)
print(f'Loop done in {(time.time() - _t_all)/60:.1f} min')

# ------------------------------------------------------------
# 6. Summary over the seeds (reads all the per-seed files)
# ------------------------------------------------------------
_all = []
for seed in NOISE_SEEDS:
    _f = os.path.join(NOISE_DIR, f'seed_{seed:03d}.csv')
    if os.path.exists(_f):
        for r in csv.DictReader(open(_f)):
            _all.append((r['model'], int(r['seed']), float(r['alpha']), int(r['k']), float(r['z_m']), float(r['r2'])))
_seeds_done = sorted({r[1] for r in _all})
print(f'\nSeeds in the summary: {len(_seeds_done)}')

_ALPHAS_ALL = [0.0] + NOISE_ALPHAS
_r2 = {}                                            # (model, alpha, k) -> {seed: r2}
for m, s, a, k, z, r in _all:
    _r2.setdefault((m, a, k), {})[s] = r

summary = []
for m in ('LR', 'FiLM'):
    for a in _ALPHAS_ALL:
        for t, k in enumerate(K_ALL):
            d = _r2.get((m, a, k), {})
            v = np.array([d[s] for s in _seeds_done if s in d])
            # paired drop: R2(alpha) - R2(clean) of the SAME seed
            d0 = _r2.get((m, 0.0, k), {})
            dv = np.array([d[s] - d0[s] for s in _seeds_done if s in d and s in d0])
            summary.append(dict(
                model=m, alpha=a, k=k, z_m=float(Z_LEVELS[k]), n=len(v),
                r2_mean=float(v.mean()) if len(v) else np.nan,
                r2_std=float(v.std(ddof=1)) if len(v) > 1 else 0.0,
                dr2_mean=float(dv.mean()) if len(dv) else np.nan,
                dr2_std=float(dv.std(ddof=1)) if len(dv) > 1 else 0.0,
                r2_lr_expected=(float(lr_expected[(a, t)]) if (m == 'LR' and a > 0)
                                else (r2_lr_clean[t] if m == 'LR' else np.nan))))

with open(os.path.join(NOISE_DIR, 'noise_r2_summary.csv'), 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(summary[0].keys())); w.writeheader(); w.writerows(summary)
with open(os.path.join(NOISE_DIR, 'noise_r2_all_seeds.csv'), 'w', newline='') as f:
    w = csv.writer(f); w.writerow(_FIELDS); w.writerows(_all)

_S = {(r['model'], r['alpha'], r['k']): r for r in summary}
print(f'\nR2 on Algerian, mean ± std over {len(_seeds_done)} seeds/realizations '
      f'(LR expected = analytic value for white noise)')
_hdr = f'{"k":>3} {"z (m)":>7} {"model":>6} ' + ' '.join(f'{("clean" if a == 0 else f"{100*a:.0f}%"):>15}' for a in _ALPHAS_ALL)
print(_hdr); print('-' * len(_hdr))
for k in K_PRED_cnn:
    for m in ('LR', 'FiLM'):
        print(f'{k:>3} {Z_LEVELS[k]:>7.1f} {m:>6} '
              + ' '.join(f'{_S[(m, a, k)]["r2_mean"]:>7.3f}±{_S[(m, a, k)]["r2_std"]:<7.3f}' for a in _ALPHAS_ALL))
    print(f'{"":>3} {"":>7} {"LR exp":>6} {"":>15} '
          + ' '.join(f'{_S[("LR", a, k)]["r2_lr_expected"]:>15.3f}' for a in NOISE_ALPHAS))
print('\nPaired drop ΔR2 = R2(alpha) - R2(clean), mean ± std')
for k in K_PRED_cnn:
    for m in ('LR', 'FiLM'):
        print(f'{k:>3} {Z_LEVELS[k]:>7.1f} {m:>6} '
              + ' '.join(f'{100*a:>3.0f}%: {_S[(m, a, k)]["dr2_mean"]:+.3f}±{_S[(m, a, k)]["dr2_std"]:.3f}'
                         for a in NOISE_ALPHAS))

# ------------------------------------------------------------
# 7. Figures
# ------------------------------------------------------------
C_LR, C_FILM = '#C2410C', '#1D4ED8'
_x = [100 * a for a in _ALPHAS_ALL]

# (a) R2 vs alpha at the K_PRED levels
fig, axs = plt.subplots(1, len(K_PRED_cnn), figsize=(4.2 * len(K_PRED_cnn), 3.8), sharex=True,
                        constrained_layout=True)
for ax, k in zip(axs, K_PRED_cnn):
    for m, c, mk in (('LR', C_LR, 's'), ('FiLM', C_FILM, 'o')):
        mu = [_S[(m, a, k)]['r2_mean'] for a in _ALPHAS_ALL]
        sd = [_S[(m, a, k)]['r2_std'] for a in _ALPHAS_ALL]
        ax.errorbar(_x, mu, yerr=sd, color=c, marker=mk, ms=6, lw=2, capsize=3, label=m)
    ax.plot(_x[1:], [_S[('LR', a, k)]['r2_lr_expected'] for a in NOISE_ALPHAS], ls='none',
            marker='x', ms=8, color='0.35', label='LR expected')
    ax.set_title(f'k={k}, z={Z_LEVELS[k]:.1f} m')
    ax.set_xlabel(r'$\alpha$ (%)'); ax.grid(alpha=0.3)
axs[0].set_ylabel(r'$R^2$ on the transition region')
axs[0].legend(frameon=False)
fig.suptitle(f'Input noise on u,v — transition region ({len(_seeds_done)} seeds)')
plt.savefig(os.path.join(NOISE_DIR, 'noise_r2_vs_alpha.png'), dpi=200, bbox_inches='tight')
plt.savefig(os.path.join(NOISE_DIR, 'noise_r2_vs_alpha.pdf'), bbox_inches='tight')
plt.show()

# (b) R2 vs depth, all 16 levels, one line per alpha (light -> dark = more noise)
_cols = ['black'] + [plt.cm.Greys(x) for x in np.linspace(0.45, 0.8, len(NOISE_ALPHAS))]
fig, axs = plt.subplots(1, 2, figsize=(9, 5), sharey=True, constrained_layout=True)
_z = [Z_LEVELS[k] for k in K_ALL]
for ax, m in zip(axs, ('LR', 'FiLM')):
    for a, c in zip(_ALPHAS_ALL, _cols):
        mu = np.array([_S[(m, a, k)]['r2_mean'] for k in K_ALL])
        sd = np.array([_S[(m, a, k)]['r2_std'] for k in K_ALL])
        ax.plot(mu, _z, color=c, lw=2, marker='o', ms=4, label='clean' if a == 0 else f'{100*a:.0f}%')
        ax.fill_betweenx(_z, mu - sd, mu + sd, color=c, alpha=0.15, lw=0)
    ax.set_title(m); ax.set_xlabel(r'$R^2$ on the transition region'); ax.grid(alpha=0.3)
axs[0].set_ylabel('z (m)'); axs[1].legend(frameon=False, title=r'$\alpha$')
plt.savefig(os.path.join(NOISE_DIR, 'noise_r2_vs_depth.png'), dpi=200, bbox_inches='tight')
plt.savefig(os.path.join(NOISE_DIR, 'noise_r2_vs_depth.pdf'), bbox_inches='tight')
plt.show()

print(f'\nSaved in {NOISE_DIR}: config.json, seed_XXX.csv, noise_r2_summary.csv, '
      f'noise_r2_all_seeds.csv, noise_r2_vs_alpha.png/pdf, noise_r2_vs_depth.png/pdf')